#### Importing Libraries

In [97]:
import pandas as pd
from IPython.display import display
import numpy as np
import os

#### Loading and Viewing CSV from output folder

In [98]:
df = pd.read_csv("C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/Compass_ESO_Data/output/Compass_Data_1_Cleaned.csv")
display(df.sample(5))

C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\lib\site-packages\IPython\core\interactiveshell.py:3166: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,Alarm Date,Alarm Time,EMS Apparatus Count,Incident Number,Incident Type,Incident Type Group,Latitude,Longitude,Shift,Station,Suppression Apparatus Count,Apparatus Type,Dispatch Total Response Time (min),Total On Scene Time (min),Total Time Alarm to Last Unit Cleared (min),Battalion
14822,2025-09-10,03:36:58,0.0,2528964,"EMS call, excluding vehicle accident with injury",300 - Rescue & EMS,35.917221,-78.987083,A,16,1.0,Engine,6.683333,9.633333,16.200000,3
13347,2025-07-01,19:18:40,0.0,2520705,"Medical assist, assist EMS crew",300 - Rescue & EMS,0.000000,0.000000,B,9,1.0,"Medical & rescue unit, other",11.433333,0.100000,11.533333,4
21388,2025-09-28,08:27:48,0.0,2530990,Alarm system sounded due to malfunction,700 - False Alarm,35.907326,-78.910309,A,18,2.0,Truck or aerial,11.450000,5.233333,14.450000,3
14334,2025-11-12,00:44:49,0.0,2536299,Alarm system sounded due to malfunction,700 - False Alarm,36.029625,-78.963577,B,10,2.0,Engine,8.950000,22.850000,31.633333,2
30005,2026-05-07,23:20:55,0.0,2614230,"Medical assist, assist EMS crew",300 - Rescue & EMS,35.924335,-78.859322,A,13,3.0,Engine,4.150000,14.833333,18.716667,4


In [99]:
df = df.drop(columns=["Alarm Date", "Alarm Time", "EMS Apparatus Count", "Incident Type", "Shift", 
                     "Suppression Apparatus Count", "Apparatus Type", "Total On Scene Time (min)",
                      "Total Time Alarm to Last Unit Cleared (min)"])
display(df.sample(5))

,Incident Number,Incident Type Group,Latitude,Longitude,Station,Dispatch Total Response Time (min),Battalion
19043,2536110,300 - Rescue & EMS,36.045185,-78.842079,1,5.250000,1
3795,2519412,300 - Rescue & EMS,0.000000,0.000000,3,4.300000,4
21053,2541637,300 - Rescue & EMS,35.925961,-78.953728,6,6.616667,3
4489,2607640,300 - Rescue & EMS,35.996876,-78.903572,1,6.100000,1
11211,2611375,600 - Good Intent Call,36.013672,-78.893021,1,NaN,1


#### Loading and Viewing CSV from input folder

In [100]:
df2 = pd.read_csv("C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/Compass_ESO_Data/input/Fire_Stations_Geocoded 1.csv")
display(df2.sample(5))

,Station,Battalion,Physical.Location,lat,long
7,Station 8,4,"225 Lick Creek Lane Durham, NC 27703",35.978626,-78.808354
15,Station 16,3,"6303 Farrington Road Chapel Hill, NC 27517",35.905729,-78.982744
11,Station 12,3,"1230 Carpenter Fletcher Road Durham, NC 27713",35.915234,-78.897464
3,Station 4,1,"1818 Riddle Road Durham, NC 27713",35.955776,-78.889658
14,Station 15,2,"2060 Torredge Road Durham, NC 27712",36.098238,-78.867640


In [101]:
df2 = df2.rename(columns={
    "lat": "Station lat",
    "long": "Station long"
})
df2 = df2.drop(columns=["Battalion"])

display(df2.sample(5))

,Station,Physical.Location,Station lat,Station long
17,Station 18,"6919 Herndon Road Durham, NC 27713",35.898550,-78.926548
13,Station 14,"1327 Umstead Road Durham, NC 27712",36.082420,-78.940218
14,Station 15,"2060 Torredge Road Durham, NC 27712",36.098238,-78.867640
11,Station 12,"1230 Carpenter Fletcher Road Durham, NC 27713",35.915234,-78.897464
18,Station 19,"4716 Old Page Road Durham, NC 27703",35.883820,-78.846571


#### df3 = df2 + df

In [102]:
df2["Station"] = df2["Station"].str.replace("Station", "", regex=False).str.strip().astype(int)
df3 = df.merge(
    df2[["Station", "Physical.Location", "Station lat", "Station long"]],
    on="Station",
    how="left"
)
display(df3.sample(5))

,Incident Number,Incident Type Group,Latitude,Longitude,Station,Dispatch Total Response Time (min),Battalion,Station lat,Station long
4375,2604712,300 - Rescue & EMS,35.984837,-78.819260,8,3.666667,4,35.978626,-78.808354
32608,2522932,300 - Rescue & EMS,0.000000,0.000000,10,8.350000,2,36.041914,-78.961552
16298,2532533,300 - Rescue & EMS,36.013653,-78.944183,2,8.300000,1,36.014123,-78.921872
9100,2609840,700 - False Alarm,35.995049,-78.918922,5,4.966667,1,35.983517,-78.930987
26726,2540835,300 - Rescue & EMS,36.128704,-78.866150,15,8.216667,2,36.098238,-78.867640


#### Average Response Time by Incident Type Group

In [103]:
df3["Dispatch Total Response Time (min)"] = pd.to_numeric(
    df3["Dispatch Total Response Time (min)"],
    errors="coerce"
)

df3["Incident Type Group Avg Response Time"] = df3.groupby(
    "Incident Type Group"
)["Dispatch Total Response Time (min)"].transform("mean")

display(df3.sample(5))

,Incident Number,Incident Type Group,Latitude,Longitude,Station,Dispatch Total Response Time (min),Battalion,Station lat,Station long,Incident Type Group Avg Response Time
7696,2603114,300 - Rescue & EMS,35.960743,-78.911598,4,4.483333,1,35.955776,-78.889658,6.154858
20850,2528335,300 - Rescue & EMS,35.939171,-78.934929,6,6.316667,3,35.933531,-78.951136,6.154858
11830,2523745,300 - Rescue & EMS,0.000000,0.000000,11,4.500000,1,35.990172,-78.968400,6.154858
8224,2538740,400 - Hazardous Condition,36.076096,-78.894203,15,8.833333,2,36.098238,-78.867640,7.321242
31899,2531489,300 - Rescue & EMS,36.050816,-78.904533,7,3.266667,2,36.054402,-78.904624,6.154858


#### Average Response Time by Station

In [104]:
df3["Dispatch Total Response Time (min)"] = pd.to_numeric(
    df3["Dispatch Total Response Time (min)"],
    errors="coerce"
)

df3["Station Avg Response Time"] = df3.groupby(
    "Station"
)["Dispatch Total Response Time (min)"].transform("mean")

display(df3.sample(5))

,Incident Number,Incident Type Group,Latitude,Longitude,Station,Dispatch Total Response Time (min),Battalion,Station lat,Station long,Incident Type Group Avg Response Time,Station Avg Response Time
24790,2526590,700 - False Alarm,35.975250,-78.902344,4,8.283333,1,35.955776,-78.889658,6.775265,6.236824
33019,2537378,300 - Rescue & EMS,36.010632,-78.918121,2,3.516667,1,36.014123,-78.921872,6.154858,6.448095
587,2518407,300 - Rescue & EMS,0.000000,0.000000,6,5.050000,3,35.933531,-78.951136,6.154858,6.551053
18676,2612325,300 - Rescue & EMS,35.990017,-78.876205,3,3.000000,4,35.990902,-78.867037,6.154858,5.358302
32210,2615750,300 - Rescue & EMS,35.992817,-78.891449,1,5.050000,1,35.996845,-78.897998,6.154858,5.547200


#### Exporting File to Output Folder

In [105]:
# Ensuring output folder exists
os.makedirs("output", exist_ok=True)

# Exporting cleaned dataframe
file_path = "C:/Users/AmarachiOny/Desktop/DFD_Projects/Python/Compass_ESO_Data/output/Compass_Data_2_Cleaned.csv"
df3.to_csv(file_path, index=False)

print("Export successful: Compass_Data_2_Cleaned.csv has been created in the output folder.")

Export successful: Compass_Data_2_Cleaned.csv has been created in the output folder.
